In [ ]:
# Example 1: Inspect the FastAPIInstrumentor primary API surface.
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

instrumentor = FastAPIInstrumentor()

# The instrumentor must expose the BaseInstrumentor lifecycle methods.
for method_name in ("instrument", "uninstrument", "instrument_app", "uninstrument_app", "instrumentation_dependencies"):
    assert hasattr(FastAPIInstrumentor, method_name), f"missing method: {method_name}"
    print(f"  FastAPIInstrumentor.{method_name}: {getattr(FastAPIInstrumentor, method_name)}")

deps = instrumentor.instrumentation_dependencies()
print(f"\ninstrumentation_dependencies() -> {list(deps)}")
assert any("fastapi" in d for d in deps), "expected a fastapi dependency spec"


In [ ]:
# Example 2: Instrument a single FastAPI app via instrument_app() and verify the wiring.
# Current releases no longer append to app.user_middleware — instrument_app() swaps in a
# wrapped build_middleware_stack builder and stashes the original on the app.
from fastapi import FastAPI
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

app = FastAPI()

@app.get("/ping")
def ping():
    return {"ok": True}

assert not hasattr(app, "_original_build_middleware_stack")
FastAPIInstrumentor.instrument_app(app)

assert getattr(app, "_is_instrumented_by_opentelemetry", False) is True
assert hasattr(app, "_original_build_middleware_stack"), "expected the wrapped middleware-stack builder"
print("instrument_app installed the OTel middleware-stack builder")

FastAPIInstrumentor.uninstrument_app(app)
assert getattr(app, "_is_instrumented_by_opentelemetry") is False
assert not hasattr(app, "_original_build_middleware_stack")
print("uninstrument_app restored the original builder")

In [ ]:
# Example 3: End-to-end span emission using an in-memory span exporter.
from fastapi import FastAPI
from fastapi.testclient import TestClient
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

app = FastAPI()

@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id}

FastAPIInstrumentor.instrument_app(app, tracer_provider=provider)
client = TestClient(app)
resp = client.get("/items/42")
assert resp.status_code == 200

spans = exporter.get_finished_spans()
print(f"captured {len(spans)} span(s); names: {[s.name for s in spans]}")
assert len(spans) > 0, "expected at least one span from the instrumented request"


In [ ]:
# Example 4: server_request_hook / client_request_hook are invoked during a request.
from fastapi import FastAPI
from fastapi.testclient import TestClient
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

calls = {"server": 0, "client_request": 0, "client_response": 0}

def server_request_hook(span, scope):
    calls["server"] += 1

def client_request_hook(span, scope, message):
    calls["client_request"] += 1

def client_response_hook(span, scope, message):
    calls["client_response"] += 1

app = FastAPI()

@app.get("/hello")
def hello():
    return {"msg": "hi"}

FastAPIInstrumentor.instrument_app(app, server_request_hook=server_request_hook, client_request_hook=client_request_hook, client_response_hook=client_response_hook)
client = TestClient(app)
assert client.get("/hello").status_code == 200
print(f"hook call counts: {calls}")
assert calls["server"] >= 1, "server_request_hook should fire at least once per request"


In [ ]:
# Example 5: excluded_urls keeps configured paths from being traced.
from fastapi import FastAPI
from fastapi.testclient import TestClient
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

provider = TracerProvider()
exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(exporter))

app = FastAPI()

@app.get("/health")
def health():
    return {"ok": True}

@app.get("/work")
def work():
    return {"done": True}

FastAPIInstrumentor.instrument_app(app, tracer_provider=provider, excluded_urls="health")
client = TestClient(app)
client.get("/health")
client.get("/work")

span_names = [s.name for s in exporter.get_finished_spans()]
print(f"span names: {span_names}")
assert any("/work" in n or "work" in n for n in span_names), "/work should produce a span"
assert not any("/health" in n for n in span_names), "/health was excluded; no span should mention it"
